## BWA alignment against Hg38 (filtered reads containing CMV primer sequence)

In [1]:
# -- Create directories for references/aligments (RUN JUST ONCE)
from pathlib import Path

# -- Create directories for references/alignments
path = Path('/Users/leandro/Desktop/github/NGS-data/ORF2library1_260411/alignments/BWA/over-hg38')

In [ ]:
# Reads pre-preprocessing (search PCR primer sequence in R1 reads)
!fqgrep "CTCCATATATGGGCTATGAACTAATGACCCC" E3-control_S1_L001_R1_001.fastq.gz > matched_reads_E3.fastq
# same for other R1 files (done in terminal)

In [3]:
# Define data location
reads_dir = path/'merged'
fastq_files = sorted(reads_dir.glob("*.merged.fastq.gz"))

fastq_files

[PosixPath('/Users/leandro/Desktop/github/NGS-data/ORF2library1_260411/merged/E3-control_S1_L001.merged.fastq.gz'),
 PosixPath('/Users/leandro/Desktop/github/NGS-data/ORF2library1_260411/merged/Hs-sorted_S2_L001.merged.fastq.gz'),
 PosixPath('/Users/leandro/Desktop/github/NGS-data/ORF2library1_260411/merged/Rr-sorted_S3_L001.merged.fastq.gz')]

In [18]:
# Basic mapping statistics

for bam in path.glob("*.bam"):
    print(f"\n===== {bam.name} =====")
    !samtools flagstat {bam}


===== Rr.bam =====
402194 + 0 in total (QC-passed reads + QC-failed reads)
382203 + 0 primary
0 + 0 secondary
19991 + 0 supplementary
0 + 0 duplicates
0 + 0 primary duplicates
304242 + 0 mapped (75.65% : N/A)
284251 + 0 primary mapped (74.37% : N/A)
0 + 0 paired in sequencing
0 + 0 read1
0 + 0 read2
0 + 0 properly paired (N/A : N/A)
0 + 0 with itself and mate mapped
0 + 0 singletons (N/A : N/A)
0 + 0 with mate mapped to a different chr
0 + 0 with mate mapped to a different chr (mapQ>=5)

===== Hs.bam =====
194790 + 0 in total (QC-passed reads + QC-failed reads)
185114 + 0 primary
0 + 0 secondary
9676 + 0 supplementary
0 + 0 duplicates
0 + 0 primary duplicates
154550 + 0 mapped (79.34% : N/A)
144874 + 0 primary mapped (78.26% : N/A)
0 + 0 paired in sequencing
0 + 0 read1
0 + 0 read2
0 + 0 properly paired (N/A : N/A)
0 + 0 with itself and mate mapped
0 + 0 singletons (N/A : N/A)
0 + 0 with mate mapped to a different chr
0 + 0 with mate mapped to a different chr (mapQ>=5)

===== E3.bam =

In [9]:
# Count uniquely mapped reads (MAPQ >0)

for bam in alignments.glob("*.bam"):
    print(f"\n===== {bam.name} =====")
    !samtools view -q 1 -c {bam}


===== Hs-sorted_S2_L001.hg38.bam =====
125906


python(82435) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.



===== E3-control_S1_L001.hg38.bam =====
233944


python(82440) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.



===== Rr-sorted_S3_L001.hg38.bam =====
231105


python(82441) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.


In [2]:
# Number of distinct genomic loci hit

import subprocess

for bam in path.glob("*.bam"):
    print(f"\n===== {bam.name} =====")
    cmd = f"samtools view {bam} | awk '{{print $3\":\"$4}}' | sort | uniq | wc -l"
    subprocess.run(cmd, shell=True)


===== Rr.bam =====
  248970

===== Hs.bam =====
  133682

===== E3.bam =====
   70834


In [7]:
# Reads per chromosome (sorted by ch number, and excluding scaffolds)

for bam in path.glob("*.bam"):
    print(f"\n===== {bam.name} =====")
    #!samtools view {bam} | cut -f3 | sort | uniq -c | sort -k2,2V # (including chromosome scaffolds)
    !samtools view {bam} | cut -f3 | grep -E '^(chr)?([1-9]|1[0-9]|2[0-2]|X|Y)$' | sort | uniq -c | sort -k2,2V


===== Rr.bam =====
31886 1
20379 2
17044 3
12096 4
14415 5
14988 6
15019 7
11530 8
11816 9
15794 10
13960 11
11604 12
6848 13
8696 14
8744 15
11687 16
14553 17
5500 18
11934 19
7919 20
5104 21
8762 22
12639 X
 962 Y

===== Hs.bam =====
16016 1
10317 2
8619 3
6400 4
7316 5
7511 6
7741 7
5996 8
6093 9
8021 10
6991 11
5847 12
3628 13
4296 14
4415 15
5890 16
6974 17
2765 18
5828 19
4001 20
2512 21
4317 22
6601 X
 548 Y

===== E3.bam =====
8322 1
5292 2
4436 3
3200 4
3849 5
3964 6
4284 7
3062 8
3330 9
4224 10
3825 11
3113 12
1687 13
2326 14
2313 15
3470 16
4275 17
1463 18
3683 19
2185 20
1591 21
2718 22
3132 X
 282 Y


In [16]:
# Alignment length on the reference 

import pysam

bam = pysam.AlignmentFile('/Users/leandro/Desktop/github/NGS-data/ORF2library1_260411/alignments/BWA/over-hg38/E3.bam')

for read in bam.head(10):
    print(read.reference_length)

160
111
49
30
126
126
119
81
111
160


In [7]:
# Summary table

import pandas as pd
import subprocess

results = []

for bam in path.glob("*.bam"):
    total = int(subprocess.check_output(
        f"samtools view -c {bam}", shell=True
    ).decode().strip())

    unique = int(subprocess.check_output(
        f"samtools view -q 30 -c {bam}", shell=True
    ).decode().strip())

    multi = int(subprocess.check_output(
        f"samtools view -c -f 256 {bam}", shell=True
    ).decode().strip())

    results.append({
        "sample": bam.name,
        "total_reads": total,
        "unique_MAPQ30": unique,
        "secondary_alignments": multi
    })

df = pd.DataFrame(results)
df

,sample,total_reads,unique_MAPQ30,secondary_alignments
0,Rr.bam,402194,241252,0
1,Hs.bam,194790,123045,0
2,E3.bam,97490,66326,0


### Manhattan plot of significance 

## BWA alignment against a1_E3 vector (filtered reads containing CMV primer sequence)

In [ ]:
chrom_levels <- c(as.character(1:22), "X", "Y")
all_counts <- list()

for (bam in bam_files) {
  cmd <- paste0(
    "samtools view ", shQuote(bam),
    " | cut -f3 | grep -E '^(chr)?([1-9]|1[0-9]|2[0-2]|X|Y)$' | sort | uniq -c | sort -k2,2V"
  )
  
  out <- system(cmd, intern = TRUE)
  
  if (length(out) == 0) next
  
  df <- read_table2(paste(out, collapse = "\n"), col_names = c("n_reads", "chromosome"))
  df$sample <- tools::file_path_sans_ext(basename(bam))
  df$chromosome <- sub("^chr", "", df$chromosome)
  df$chromosome <- factor(df$chromosome, levels = chrom_levels)
  
  all_counts[[length(all_counts) + 1]] <- df
}

counts_df <- bind_rows(all_counts)

ggplot(counts_df, aes(x = chromosome, y = n_reads, fill = sample)) +
  geom_col(position = "dodge") +
  scale_x_discrete(drop = FALSE) +
  labs(x = "Chromosome", y = "Number of reads", fill = "Sample") +
  theme_minimal(base_size = 12) +
  theme(axis.text.x = element_text(angle = 45, hjust = 1))

In [1]:
# -- Create directories for references/aligments 
from pathlib import Path

# -- Create directories for references/alignments
path = Path('/Users/leandro/Desktop/github/NGS-data/ORF2library1_260411/alignments/BWA/over-a1E3')

In [4]:
# Basic mapping statistics

for bam in path.glob("*.bam"):
    print(f"\n===== {bam.name} =====")
    !samtools flagstat {bam}


===== Rr.bam =====
382742 + 0 in total (QC-passed reads + QC-failed reads)
382203 + 0 primary
0 + 0 secondary
539 + 0 supplementary
0 + 0 duplicates
0 + 0 primary duplicates
382742 + 0 mapped (100.00% : N/A)
382203 + 0 primary mapped (100.00% : N/A)
0 + 0 paired in sequencing
0 + 0 read1
0 + 0 read2
0 + 0 properly paired (N/A : N/A)
0 + 0 with itself and mate mapped
0 + 0 singletons (N/A : N/A)
0 + 0 with mate mapped to a different chr
0 + 0 with mate mapped to a different chr (mapQ>=5)

===== Hs.bam =====
185381 + 0 in total (QC-passed reads + QC-failed reads)
185114 + 0 primary
0 + 0 secondary
267 + 0 supplementary
0 + 0 duplicates
0 + 0 primary duplicates
185381 + 0 mapped (100.00% : N/A)
185114 + 0 primary mapped (100.00% : N/A)
0 + 0 paired in sequencing
0 + 0 read1
0 + 0 read2
0 + 0 properly paired (N/A : N/A)
0 + 0 with itself and mate mapped
0 + 0 singletons (N/A : N/A)
0 + 0 with mate mapped to a different chr
0 + 0 with mate mapped to a different chr (mapQ>=5)

===== E3.bam 

In [2]:
# Number of distinct genomic loci hit

import subprocess

for bam in path.glob("*.bam"):
    print(f"\n===== {bam.name} =====")
    cmd = f"samtools view {bam} | awk '{{print $3\":\"$4}}' | sort | uniq | wc -l"
    subprocess.run(cmd, shell=True)


===== Rr.bam =====
      64

===== Hs.bam =====
      50

===== E3.bam =====
      61


In [16]:
# Reads per chromosome

for bam in path.glob("*.bam"):
    print(f"\n===== {bam.name} =====")
    !samtools view {bam} | cut -f3 | sort | uniq -c


===== Rr.bam =====
382742 a1_E3

===== Hs.bam =====
185381 a1_E3

===== E3.bam =====
92810 a1_E3


In [3]:
# Summary table

import pandas as pd
import subprocess

results = []

for bam in path.glob("*.bam"):
    total = int(subprocess.check_output(
        f"samtools view -c {bam}", shell=True
    ).decode().strip())

    unique = int(subprocess.check_output(
        f"samtools view -q 30 -c {bam}", shell=True
    ).decode().strip())

    multi = int(subprocess.check_output(
        f"samtools view -c -f 256 {bam}", shell=True
    ).decode().strip())

    results.append({
        "sample": bam.name,
        "total_reads": total,
        "unique_MAPQ30": unique,
        "secondary_alignments": multi
    })

df = pd.DataFrame(results)
df

,sample,total_reads,unique_MAPQ30,secondary_alignments
0,Rr.bam,382742,431,0
1,Hs.bam,185381,226,0
2,E3.bam,92810,119,0
